In [1]:
!pip3 install fastmcp --break-system-packages

MCP servers can do for LLMs what app stores did for phones: they turn models into action-taking assistants (for example, booking hotels/flights or retrieving coding info).  
In this project, you’ll connect your app/LLM to MCP servers using transport protocols: **STDIO** for local servers and **HTTP** for remote servers.

<pre style="text-align: center;">
Client or Application  ←→  [transport]  ←→  MCP Server
   (LLM)                 (Bridge)         (Local/Remote)
</pre>

After connection, usage is the same regardless of transport.

##### STDIO Review
STDIO or Standard Input/Output is three communication channels that every program automatically gets. This was originally designed for Linux/Unix systems but is now used across all operating systems. This is for MCP servers that run locally.

##### STDOUT 

```STDOUT``` (Standard Output) – The default stream where a program writes its normal output (usually your screen/terminal). We can print output using ```print()``` 

We can sketch the process. 

My Python Program: my_code = "hello" → print() →` [STDOUT] → Screen shows: hello

In [4]:
my_code="hello"
print(my_code)

import sys
sys.stdout.write("Hello")

hello
Hello

5

### STDIN

```STDIN``` (Standard Input) - Where input comes FROM (like your keyboard). Here we call the function input() with the prompt "What is your name?" You will then be prompted to input your name. The input from your keyboard will be stored in the variable name.

We can sketch the process. We see that there are several steps:
<pre>
My Python Program → [STDOUT] → You (screen)
      input()      "What is your name?"
         ↓
You (keyboard) → [STDIN] → My Python Program  
    "Alice"                    stores in 'name'
         ↓
My Python Program → [STDERR] → You (screen)
     sys.exit()    "Notebook exited after input."
</pre>
We can see the varable ```name``` has the input value.

In [5]:
name = input("What is your name?")

sys.exit("Notebook exited after input.")

SystemExit: Notebook exited after input.

/opt/homebrew/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [6]:
print(name)

test


### STDERR

```STDERR``` (standard error) is reserved for error messages and diagnostics. Both usually appear on the screen, but since they are separate streams, you can redirect them independently (e.g., send results to a file while keeping errors visible).

In [7]:
# Normal output goes to stdout
print("This is standard output (stdout)")

# Error messages can be written to stderr
print("This is an error message (stderr)", file=sys.stderr)

# Example: catching a real error
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Error occurred: {e}", file=sys.stderr)

This is standard output (stdout)


This is an error message (stderr)
Error occurred: division by zero


### Context7

In this lab, you’ll connect to the [Context7](https://context7.com/) MCP server using both **STDIO** and **HTTP** transports. Context7 provides up-to-date coding documentation for LLMs and code assistants.

You will:
- create `Client` instances  
- list available tools  
- call tools to retrieve library docs  

##### Input
- Library names (for example, `"fastmcp"`)  
- Documentation search queries  
- Requests for specific API/library info  

##### Output
- Current LLM-ready documentation  
- Library compatibility details  
- Code examples and usage patterns  
- API references and function descriptions  

##### STDIO Transport

With `StdioTransport`, everything runs locally. Your Python app starts the Context7 server as a subprocess (`npx @upstash/context7-mcp`) and communicates over stdin/stdout.

<pre style="text-align: center;">
Your Python Code  ←→  [stdio_transport]  ←→  Context7 Server (local)
    (Client)              (Bridge)             (your machine)
</pre>

`npx` downloads/runs the server on demand, while the transport layer handles message flow between your client and server.

In [8]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport, StdioTransport

stdio_transport = StdioTransport(
    command = "npx",
    args=["-y", "@upstash/context7-mcp"]
)
print(stdio_transport )

/opt/homebrew/lib/python3.13/site-packages/pydantic/plugin/_loader.py:50: UserWarning: ImportError while loading the `logfire-plugin` Pydantic plugin, this plugin will not be installed.

ImportError("cannot import name 'ReadableLogRecord' from 'opentelemetry.sdk._logs' (/opt/homebrew/lib/python3.13/site-packages/opentelemetry/sdk/_logs/__init__.py)")
  warnings.warn(


<StdioTransport(command='npx', args=['-y', '@upstash/context7-mcp'])>


In [10]:
stdio_client = Client(stdio_transport)

async with stdio_client as client:
    # List of tools the server provides
    tools = await client.list_tools()

print("Done")
len(tools)

Done


2

In [11]:
tools

[Tool(name='resolve-library-id', title='Resolve Context7 Library ID', description="Resolves a package/product name to a Context7-compatible library ID and returns matching libraries.\n\nYou MUST call this function before 'Query Documentation' tool to obtain a valid Context7-compatible library ID UNLESS the user explicitly provides a library ID in the format '/org/project' or '/org/project/version' in their query.\n\nEach result includes:\n- Library ID: Context7-compatible identifier (format: /org/project)\n- Name: Library or package name\n- Description: Short summary\n- Code Snippets: Number of available code examples\n- Source Reputation: Authority indicator (High, Medium, Low, or Unknown)\n- Benchmark Score: Quality indicator (100 is the highest score)\n- Versions: List of versions if available. Use one of those versions if the user provides a version in their query. The format of the version is /org/project/version.\n\nFor best results, select libraries based on name match, source r

##### `call_tool`

`call_tool` is the method you use to actually run/execute a specific tool on the MCP server. 

The pramaters are:

- **`tool_name`** - The tool name comes from `.name` like `tools[0].name`, which gives you "resolve-library-id"

- **`{parameters}`** - The inputs the tool needs (like `{"libraryName": "fastmcp", "query":"[YOUR QUERY]"}`). The input structure comes from `.inputSchema`

- **`response`** - What the tool sends back to you

##### `resolve-library-id`

Let's get information about the FastMCP library. We need the library identification, so we'll use the `resolve-library-id` tool. This will allow us to get the library information. The input parameter will be libraryName: "fastmcp".


In [12]:
async with stdio_client as client:
    # Find a library ID via a search query
    response = await client.call_tool("resolve-library-id", {
        "libraryName": "fastmcp",
        "query": "I want to create a new MCP server using the fastmcp Python framework"
    })

print(response.content[0].text)

Available Libraries:

- Title: FastMCP
- Context7-compatible library ID: /prefecthq/fastmcp
- Description: FastMCP is a Python framework for building Model Context Protocol (MCP) servers, clients, and apps that connect LLMs to tools, resources, and data with automatic schema generation and protocol management.
- Code Snippets: 3782
- Source Reputation: High
- Benchmark Score: 86.8
----------
- Title: FastMCP
- Context7-compatible library ID: /jlowin/fastmcp
- Description: FastMCP is a Python framework for building Model Context Protocol (MCP) servers and clients, simplifying the creation of LLM-integrated applications with features like deployment, authentication, and dynamic tool generation.
- Code Snippets: 264
- Source Reputation: High
- Benchmark Score: 69.01
- Versions: v3.0.0b1, v2.14.4, v3.0.0b2, v2.14.5, v3.0.0rc2, v3.0.1, v3.0.2, v3.1.0
----------
- Title: FastMCP
- Context7-compatible library ID: /llmstxt/gofastmcp_llms-full_txt
- Description: FastMCP is a Python framework fo

##### `query-docs`

Now we'll use the `query-docs` tool to retrieve the actual documentation for that specific library.


We'll use the context manager once again to fetch the code snippets and documentation of `/llmstxt/gofastmcp_llms-full_txt` using the `query-docs` tool.


In [13]:
async with stdio_client as client:
    # Use resolved ID to fetch documentation
    docs = await client.call_tool("query-docs", {
        "libraryId": "/llmstxt/gofastmcp_llms-full_txt",
        "query": "I want to fetch the code snippets and the documentation",
        "tokens": 5000
    })

    print(docs.content[0].text[:1000])

### Fetch All Tools Automatically (JavaScript)

Source: https://gofastmcp.com/servers/pagination

This snippet demonstrates how to fetch all available tools using the Gofastmcp client. It automatically handles pagination to retrieve all 200 tools. The output includes the total number of tools fetched.

```javascript
import { Client } from "fastmcp"

const client = new Client()

// Returns all 200 tools, fetching pages automatically
const tools = await client.list_tools()
print(f"Total tools: {len(tools)}")
```

--------------------------------

### Client Fetches All Paginated Items Automatically

Source: https://gofastmcp.com/servers/pagination

Demonstrates how the FastMCP Client transparently handles pagination. Convenience methods like `list_tools()` automatically fetch all pages and return the complete list, ensuring existing code functions without modification.

```python
from fastmcp import Client

async with Client(server) as client:
    # Returns all 200 tools, fetching pages 

###  Question 1.  How do you use the resolve-library-id tool to find the library ID for scikit-learn

In [14]:
async with stdio_client as client:
   # Find a library ID via a search query
   response = await client.call_tool("resolve-library-id", {
       "libraryName": "scikit-learn",
       "query": "I want to use scikit-learn package"
   })
   print(response.content[0].text[:1500])


Available Libraries:

- Title: Scikit-learn
- Context7-compatible library ID: /scikit-learn/scikit-learn
- Description: scikit-learn: machine learning in Python
- Code Snippets: 2594
- Source Reputation: High
- Benchmark Score: 80.18
- Versions: 1.7.1
----------
- Title: scikit-learn
- Context7-compatible library ID: /websites/scikit-learn_stable
- Description: scikit-learn is an open-source Python library providing simple and efficient tools for predictive data analysis, including algorithms for classification, regression, clustering, dimensionality reduction, and model selection.
- Code Snippets: 28571
- Source Reputation: High
- Benchmark Score: 76.1
----------
- Title: scikit-learn
- Context7-compatible library ID: /websites/scikit-learn_dev
- Description: scikit-learn is a free and open-source Python machine learning library that provides simple and efficient tools for data analysis, predictive modeling, and statistical learning.
- Code Snippets: 29216
- Source Reputation: High
- 

### Question 2. How do you get the actual documentation once you have the library ID?

In [15]:
async with stdio_client as client:
    # Use resolved ID to fetch documentation
    docs = await client.call_tool("query-docs", {
        "libraryId": "/scikit-learn/scikit-learn",
        "query": "I want to fetch the documentation of the package.",
        "tokens": 5000
    })
    print(docs.content[0].text[:1000])

### Build and Serve Documentation Locally

Source: https://github.com/scikit-learn/scikit-learn/blob/main/doc/developers/contributing.rst

Shell commands to install the necessary documentation dependencies, build the HTML output using make, and serve the generated site locally.

```bash
pip install sphinx sphinx-gallery numpydoc matplotlib Pillow pandas \
            polars scikit-image packaging seaborn sphinx-prompt \
            sphinxext-opengraph sphinx-copybutton plotly pooch \
            pydata-sphinx-theme sphinxcontrib-sass sphinx-design \
            sphinx-remove-toctrees

cd doc
make
make html

python -m http.server -d _build/html
```

--------------------------------

### Build Documentation Commands

Source: https://github.com/scikit-learn/scikit-learn/blob/main/doc/developers/contributing.rst

Commands to build specific documentation examples or the full PDF manual using make.

```bash
EXAMPLES_PATTERN="plot_calibration" make html
make latexpdf
```

--------------------

### HTTP Preface

The HTTP protocol allows data to be transferred over the internet. Since the protocol can be somewhat complex, we will focus on the GET method, one of the most common HTTP methods.

The figure below illustrates a typical response. The response start line contains:

- Version number (e.g., HTTP/1.0)

- Status code (e.g., 200 for success)

- Descriptive phrase (e.g., OK)

The response header follows, providing useful metadata. Finally, the response body includes the requested resource, such as an HTML document. It should also be noted that some requests include headers, which provide additional details or parameters for the transaction.

Requests is a Python Library that allows you to send `HTTP/1.1` requests easily. We can import the library as follows:

In [16]:
import requests

url='https://www.ibm.com/'
r=requests.get(url)
r.status_code

200

In [17]:
print(r.request.headers)
print("request body:", r.request.body)

{'User-Agent': 'python-requests/2.32.5', 'Accept-Encoding': 'gzip, deflate, br', 'Accept': '*/*', 'Connection': 'keep-alive', 'Cookie': '_abck=91BF47FEBFD9CBC14D37370F7E54470F~-1~YAAQXZ42F28YsuGcAQAAp9RU8Q/D1krnr8w+0ObNclnU38040qpn0uGr8NNuvD5LjVQeN1Mo41esLO//bvdQShaMBbencVzem2VaAwXNu9GxArEJXwBuTfDisqryfaIzgfzFW2mbBWiBtrYaEsF/H2lBKcbS7XVNryLrd0xQjw1WRw8huvBd1fdPWzRb/isok6FaAEKZFtDVdZuZO5BuanA85SziFqFYuS3Dnst6Uk7E+N1OxGA0+uq6dHajzqjVBv5PJ6JdQKMopG3T6j80xnlj3aFfsSQTQKRA3LQ6YLo1OEgDOonzdE/Skz17xbDiYpz13Vr6CZjxsUTkWoW9LNhvN8hu2FY2fg5sCUw+3XLPCyToGM3qHdALx0co8RypHB67OoBFXtlVjO0YzeLQaNwdPyJqFOpZl6IErCFvtd6/7dHmPUZ3WxV4eTBtNiUOUi4=~-1~-1~-1~-1~-1; bm_sz=F385327EABA6E1A007B3C7F6645D6982~YAAQXZ42F3AYsuGcAQAAp9RU8R/5q3A7PyPqdEPWz+DmosYHfZcySQOdFfP//0PS4K5g7a7+GYQ8/ImD1lNG2gNsko+/+2WZbesLP0JJNCkapyBueys8t+Kz9xaUtUwOvAQyIgiEBRA6HrGKukOX+jhVCfVp3hKzOQaXhnCnBEbSfCqyZdyRgKuftlBakqToEDHN5zy7Y6YaPzae3d5SgijhAnnSBJz++zSbl3csH0kAwFPzAk+anqrMNiC+VH+9LKi59YsFEsjZKhh4767ZZ64LoTEdezQQB+acWIqCQz6IRiDhPrSSbNeXR

##### HTTP Transport

Now let's talk to the Context7 MCP server over HTTPS—no local process needed. The HTTP transport also acts like a bridge, but instead of connecting from your notebook/Python file to your local computer, it connects to a remote server. This is ideal when the server is hosted remotely or you don't want to manage a subprocess. We'll use the same Client API with a different "wire" (transport). Once you create the transport, the process is pretty much identical.

<pre style="text-align: center;">
Your Python Code  ←→  [http_transport]  ←→  Context7 Server (remote)
   (Client)              (Bridge)              (Context7's servers)
</pre>

In [18]:
http_transport = StreamableHttpTransport(
    url="https://mcp.context7.com/mcp"
)

http_client = Client(http_transport)

async with http_client as client:
    tools = await client.list_tools()

    response = await client.call_tool("resolve-library-id", {
        "libraryName": "fastmcp",
        "query": "I want to create a new MCP server using the fastmcp Python framework"
    })

    docs = await client.call_tool("query-docs", {
        "libraryId": "/llmstxt/gofastmcp_llms-full_txt",
        "query": "I want to fetch the code snippets and the documentation",
        "tokens": 5000
    })

In [19]:
for tool in tools:
        print(
f"""{tool.name}: \n
{tool.description} \n
{tool.inputSchema}""")
print(response.content[0].text[:1000])
print(docs.content[0].text[:500]) 

resolve-library-id: 

Resolves a package/product name to a Context7-compatible library ID and returns matching libraries.

You MUST call this function before 'Query Documentation' tool to obtain a valid Context7-compatible library ID UNLESS the user explicitly provides a library ID in the format '/org/project' or '/org/project/version' in their query.

Each result includes:
- Library ID: Context7-compatible identifier (format: /org/project)
- Name: Library or package name
- Description: Short summary
- Code Snippets: Number of available code examples
- Source Reputation: Authority indicator (High, Medium, Low, or Unknown)
- Benchmark Score: Quality indicator (100 is the highest score)
- Versions: List of versions if available. Use one of those versions if the user provides a version in their query. The format of the version is /org/project/version.

For best results, select libraries based on name match, source reputation, snippet coverage, benchmark score, and relevance to your use ca

### Other Transports

There are other transports but we won't demo them here as they are not as widely used as STDIO or HTTP. These two are enough for most use cases of MCP servers.

##### SSE Transport

The Server-Sent Events transport enables HTTP communication between MCP servers and clients using an asymmetric channel. It uses SSE for server-to-client streaming and HTTP POST for client-to-server messages. It was replaced by the Streamable HTTP transport to provide bidirectional streaming capabilities and improved real-time communication efficiency.

Read more [here](https://gofastmcp.com/clients/transports#sse-transport-legacy).

##### In-Memory Transport

In-memory transport connects a client directly to a FastMCP server instance within the same Python process. This eliminates network overhead by enabling direct function calls between client and server components. 

Read more [here](https://gofastmcp.com/clients/transports#in-memory-transport).
